In [3]:
import pandas as pd
import numpy as np
import time
from openpyxl import load_workbook

# Read input data
data = pd.read_excel('/Users/workk/Documents/[Think!]/[FMK]/Research Things/Economic Dispatch/[Economic Emission Dispatch (EED)]/Datasets/Jamali/EED8u.xlsx')
a = data['a'].values
b = data['b'].values
c = data['c'].values
d = data['d'].values
e = data['e'].values
f = data['f'].values
Minimum_Capacity = data['pmin'].values
Maximum_Capacity = data['pmax'].values
ramp_up = data['rampup'].values
ramp_down = data['rampdown'].values
Unit = data['Unit'].values

# Read power demand data
power_demand_data = pd.read_excel('/Users/workk/Documents/[Think!]/[FMK]/Research Things/Economic Dispatch/[Economic Emission Dispatch (EED)]/Datasets/Jamali/EED_demand8u.xlsx')
power_demand = power_demand_data['load2'].values

# Initialize an empty list to store results for each demand
all_outputs = []

# Define function to calculate fuel cost
def calculate_fuel_cost(P, a, b, c):
    squaredP = np.multiply(P, P)
    Fuel_Cost = np.divide(np.add(a, np.multiply(b, P) + np.multiply(c, squaredP)), (14385))
    return np.sum(Fuel_Cost)

# Define function to calculate emissions
def calculate_emissions(P, d, e, f):
    Emissions = np.divide(np.add(np.add(d, np.multiply(e, P)), np.multiply(f, np.power(P, 2))), 1e6)
    return np.sum(Emissions)

# Objective function: weighted sum of fuel cost and emissions
def objective_function(P, a, b, c, d, e, f, demand, penalty_factor, w1, w2):
    fuel_cost = calculate_fuel_cost(P, a, b, c)
    emissions = calculate_emissions(P, d, e, f)
    power_difference = np.sum(P) - demand
    penalty_term = penalty_factor * abs(power_difference)
    return w1 * fuel_cost + w2 * emissions + penalty_term

# Constraints: total power production must meet the demand
def power_balance_constraint(P, demand):
    return np.sum(P) - demand

# Boundary constraints for generation limits
bounds = [(Minimum_Capacity[i], Maximum_Capacity[i]) for i in range(len(Minimum_Capacity))]

# Backtracking line search function
def backtracking_line_search(P, grad, func, alpha=0.3, beta=0.8, tol=1e-6):
    step = 1.0
    while step > tol:
        new_P = P - step * grad
        if func(new_P) < func(P) - alpha * step * np.dot(grad, grad):
            return new_P
        step *= beta
    return P

# GRASP-based Hyper-heuristic with Backtracking Line Search
def grasp_heuristic(func, bounds, max_iterations=100, alpha=0.3, tol=1e-6):
    n = len(bounds)
    best_solution = None
    best_value = float('inf')

    for iteration in range(max_iterations):
        # Constructive Phase
        P = np.zeros(n)
        for i in range(n):
            candidates = np.linspace(bounds[i][0], bounds[i][1], 100)
            np.random.shuffle(candidates)
            candidates = candidates[:int(alpha * len(candidates))]
            best_candidate = None
            best_candidate_value = float('inf')
            for candidate in candidates:
                P[i] = candidate
                value = func(P)
                if value < best_candidate_value:
                    best_candidate_value = value
                    best_candidate = candidate
            P[i] = best_candidate

        # Local Search Phase with Backtracking Line Search
        step = 1.0
        while step > tol:
            grad = np.zeros(n)
            for i in range(n):
                delta = np.zeros(n)
                delta[i] = 1e-8
                grad[i] = (func(P + delta) - func(P - delta)) / (2 * 1e-8)
            new_P = backtracking_line_search(P, grad, func, alpha=0.3, beta=0.8, tol=1e-6)
            if np.allclose(new_P, P, atol=tol):
                break
            P = new_P
            step *= 0.5

        value = func(P)
        if value < best_value:
            best_value = value
            best_solution = P

    return best_solution

# Enforce ramp rate limits
def enforce_ramp_limits(previous_output, current_output, ramp_up, ramp_down):
    adjusted_output = np.copy(current_output)
    ramp_rate = current_output - previous_output

    # Apply ramp up/down constraints
    for i in range(len(adjusted_output)):
        if ramp_rate[i] > ramp_up[i]:
            adjusted_output[i] = previous_output[i] + ramp_up[i]  # Limit ramp up
        elif ramp_rate[i] < -ramp_down[i]:
            adjusted_output[i] = previous_output[i] - ramp_down[i]  # Limit ramp down

    return adjusted_output

# Optimization process for each demand
penalty_factor = 1e6
tol = 1e-3

# List of weighting scenarios
scenarios = [(1, 0)]

# Loop through each scenario
all_scenario_results = []

for scenario_index, (w1, w2) in enumerate(scenarios, start=1):
    print(f"Running scenario {scenario_index}: w1={w1}, w2={w2}")
    all_outputs = []
    schedulling = []

    for load_counter, demand in enumerate(power_demand, start=1):
        start_time = time.time()
        if load_counter == 1:
            previous_output = None
        else:
            previous_output = all_outputs[-1]['Output']['Power Produced (MW)'].values

        # Initial guess for optimization
        P = np.random.uniform(low=Minimum_Capacity, high=Maximum_Capacity, size=len(Minimum_Capacity))

        def func_to_minimize(P):
            return objective_function(P, a, b, c, d, e, f, demand, penalty_factor, w1, w2)

        P = grasp_heuristic(func_to_minimize, bounds, tol=tol)

        # Ensure final solution adheres to generation limits
        P = np.minimum(np.maximum(P, Minimum_Capacity), Maximum_Capacity)

        # Force power balance if violated
        power_difference = np.sum(P) - demand
        while abs(power_difference) > tol:
            adjustment_per_unit = power_difference / len(P)
            P -= adjustment_per_unit
            P = np.minimum(np.maximum(P, Minimum_Capacity), Maximum_Capacity)
            power_difference = np.sum(P) - demand

        end_time = time.time()

        # Extract results
        P_optimal = P
        fuel_cost = calculate_fuel_cost(P_optimal, a, b, c)
        emissions = calculate_emissions(P_optimal, d, e, f)

        # Output Results
        output = pd.DataFrame({'Unit': Unit, 'Power Produced (MW)': P_optimal, 'Emissions (Tons)': emissions})
        print("Load Counter: {}".format(load_counter))
        print(f"Output for Power Demand {demand:.2f} MW")
        print(output)
        print(f"Total Power Produced: {np.sum(P_optimal):.2f} MW")
        print(f'Total Fuel Cost: Rp {fuel_cost}')
        print('Total Emissions (Tons): ', emissions)
        print(f'Computation Time: {end_time - start_time} seconds')
        print('------------------------------------------------\n')

        all_outputs.append({'Demand': demand, 
                            'Output': output, 
                            'Total Power Produced': np.sum(P_optimal),
                            'Total Fuel Cost': fuel_cost,  
                            'Total Emissions (Tons)': emissions,
                            'Computation Time': end_time - start_time})

        # Scheduling
        output_info = {'Demand': power_demand[load_counter - 1]}
        for unit_index, unit in enumerate(Unit):
            output_info[f'Unit {unit} Power Produced'] = P_optimal[unit_index]
            output_info[f'Unit {unit} Cost'] = fuel_cost  
            output_info[f'Unit {unit} Emissions'] = emissions  

        schedulling.append(output_info)

    all_scenario_results.append((all_outputs, schedulling))

# Save results for all scenarios to Excel
output_file_path = '/Users/workk/Documents/[Think!]/[FMK]/Research Things/Economic Dispatch/Output/GRASP-BLS (EED).xlsx'
with pd.ExcelWriter(output_file_path) as writer:
    for scenario_index, (all_outputs, schedulling) in enumerate(all_scenario_results, start=1):
        # Calculate ramp rates and generation limits for the current scenario
        ramp_rates = []
        generation_limits = []
        for i in range(1, len(power_demand)):  # Start from 1 because there's no previous output for the first demand
            previous_output = all_outputs[i-1]['Output']['Power Produced (MW)'].values
            current_output = all_outputs[i]['Output']['Power Produced (MW)'].values

            # Enforce ramp rate limits
            adjusted_output = enforce_ramp_limits(previous_output, current_output, ramp_up, ramp_down)

            ramp_rate = adjusted_output - previous_output

            # Check for ramp rate violations
            violations = np.logical_or(ramp_rate > ramp_up, ramp_rate < -ramp_down)
            ramp_rate_info = {'Demand': power_demand[i]}
            generation_limit_info = {'Demand': power_demand[i]}
            for unit_index, unit in enumerate(Unit):
                ramp_rate_info[f'Unit {unit}'] = ramp_rate[unit_index]
                ramp_rate_info[f'Unit {unit} Violation'] = violations[unit_index]

                # Check for generation limit violations after adjusting output
                gen_violation = (adjusted_output[unit_index] < Minimum_Capacity[unit_index]) or (adjusted_output[unit_index] > Maximum_Capacity[unit_index])
                generation_limit_info[f'Unit {unit}'] = adjusted_output[unit_index]
                generation_limit_info[f'Unit {unit} Violation'] = gen_violation

            ramp_rates.append(ramp_rate_info)
            generation_limits.append(generation_limit_info)

            # Update the optimal power output to reflect ramp rate adjustments
            all_outputs[i]['Output']['Power Produced (MW)'] = adjusted_output

        # Save scenario results to Excel
        pd.DataFrame(schedulling).to_excel(writer, sheet_name=f'Scenario {scenario_index} Schedulling', index=False)
        pd.DataFrame(all_outputs).to_excel(writer, sheet_name=f'Scenario {scenario_index} Summary', index=False)
        pd.DataFrame(ramp_rates).to_excel(writer, sheet_name=f'Scenario {scenario_index} Ramp Rates', index=False)
        pd.DataFrame(generation_limits).to_excel(writer, sheet_name=f'Scenario {scenario_index} Generation Limits', index=False)

print("All results saved to 'GRASP-BLS (EED).xlsx'")


Running scenario 1: w1=1, w2=0
Load Counter: 1
Output for Power Demand 6432.50 MW
   Unit  Power Produced (MW)  Emissions (Tons)
0     1          2293.500995       8562.137846
1     2           934.000000       8562.137846
2     3           404.000000       8562.137846
3     4           208.000000       8562.137846
4     5           848.000000       8562.137846
5     6          1080.000000       8562.137846
6     7           360.000000       8562.137846
7     8           305.000000       8562.137846
Total Power Produced: 6432.50 MW
Total Fuel Cost: Rp 1232463.6776119717
Total Emissions (Tons):  8562.137845544621
Computation Time: 1.1487791538238525 seconds
------------------------------------------------

Load Counter: 2
Output for Power Demand 6127.80 MW
   Unit  Power Produced (MW)  Emissions (Tons)
0     1          1988.800969       8266.522352
1     2           934.000000       8266.522352
2     3           404.000000       8266.522352
3     4           208.000000       8266.522352